# 📊 MESA Benchmark Suite — Zero-Cost Colab Mode

This notebook runs the official MESA Evaluation Suite (`mesa_evals`) entirely on Google Colab's free tier, using **Ollama** for zero-cost LLM calls.

It tests 5 different retrieval architectures against the MESA Golden Dataset to measure Latency, Token Cost, and Accuracy (Recall).

---
## 1. Install MESA and Dependencies

In [ ]:
!apt-get update
!apt-get install -y zstd

# Clone the repository
!git clone https://github.com/Yasou13/MESA.git
%cd MESA

# Install MESA with ML and Adapter extensions
!pip install -e .[ml,adapters]

---
## 2. Install and Start Ollama
We use `llama3.2:3b` as the evaluator/benchmark LLM to run fast on Colab CPUs/GPUs.

In [ ]:
import os
import time

# Configure Zero-Cost Mode
os.environ["MESA_ZERO_COST_MODE"] = "true"
os.environ["MESA_LLM_PROVIDER"] = "ollama"
os.environ["LLM_MODEL_NAME"] = "llama3.2:3b"
os.environ["MESA_REBEL_ENABLED"] = "false"
os.environ["MESA_OLLAMA_URL"] = "http://127.0.0.1:11434"

# Dummy key so Mem0 doesn't complain during benchmark initialization
os.environ["OPENAI_API_KEY"] = "sk-dummy"

# Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(5)  # Wait for API to boot
print('\u2705 Ollama server started')

# Pull the model
!ollama pull llama3.2:3b
print('\u2705 Model ready')

---
## 3. Run the MESA Evaluation Suite
This will generate the synthetic evaluation dataset and run the 5 retrieval paths (Base, Vector, Graph, Hybrid, Hybrid_FTS5).

In [ ]:
# Execute the benchmark orchestrator
!python -m mesa_evals

---
## 4. Visualise the Results
Read the JSON output and display the metrics in a clean DataFrame.

In [ ]:
import json
import pandas as pd
from IPython.display import display

# Load the benchmark results
try:
    with open('eval_results.json', 'r') as f:
        data = json.load(f)

    # Convert summaries to a DataFrame
    df = pd.DataFrame(data['summaries']).T

    # Reorder columns for readability
    df = df[['total_entries', 'mean_recall', 'mean_ttft_ms', 'p95_ttft_ms', 'total_input_tokens', 'total_output_tokens']]
    
    print("\n\ud83c\udfc6 MESA Evaluation Summary:\n")
    display(df)
except FileNotFoundError:
    print("\u26a0\ufe0f Benchmark results not found. Make sure Step 3 completed successfully.")